# EDEN-05 — Climate model | v1.0
**Provisional Phase 2 · CPU · Internet Off · Run in Kaggle**

Learn a simple one-hour air-temperature forecast from the previous six hours of temperature readings. Compare it with persistence (repeat the latest reading) on exactly the same validation examples. **Ridge regression** learns a weighted combination of readings, with a penalty that discourages overly large weights. No GPU is needed for this small model.

**Attach:** EDEN-01 v1.1 PASS_ETL results (`results_1.1.zip` or extracted output) **and** EDEN-04 v1.0 results. The notebook finds both by their manifests, verifies all files, and checks their lineage.

**Why rebuild windows?** EDEN-04's yesterday baseline used 23 hours of history at the forecast origin, exceeding the frozen six-hour limit. Its history could also cross chronological split boundaries. Those saved scores are diagnostic only. This notebook checks the whole input-to-target interval and recalculates persistence on matching examples.

**Output:** `/kaggle/working/EDEN-05-Climate-Model-Results-v1.0.zip`, containing validation predictions, metrics, frozen model parameters, coverage and manifest. Test scoring is disabled by default. EDEN-04 test summaries have already been viewed; do not describe the test as completely unseen.

The full SRS is still unavailable. Notebook allocation and this small model's scope are working choices, not a claim that the full forecasting phase is complete. EDEN-03 image QC remains open.

## Cell 1 — Settings and tools
Keep the default settings for the first run. A **lag** is an earlier reading: lag 12 means one hour ago. We require all 72 input readings, even though the model uses only selected lags and two averages.

In [ ]:
from pathlib import Path, PurePosixPath
import io, json, hashlib, zipfile, platform, importlib.metadata
import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from IPython.display import display

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
# A fresh run directory prevents stale files entering the manifest; old runs stay intact.
RUN = pd.Timestamp.now(tz='UTC').strftime('%Y%m%dT%H%M%S%fZ')
ROOT = WORK / ('eden05_' + RUN)
for name in ('data', 'reports', 'models'):
    (ROOT / name).mkdir(parents=True, exist_ok=True)
TARGET = 'air_temp_c'
LOOKBACK, HORIZON = 72, 12
LAGS = [0, 1, 3, 6, 12, 24, 36, 48, 71]
ALPHAS = [0.1, 1.0, 10.0, 100.0]
SEED = 42
EVALUATE_TEST = False  # Keep False during model development; freeze choices first.
FEATURES = [f'lag_{lag}' for lag in LAGS] + ['mean_12', 'mean_72']
RESULT = WORK / 'EDEN-05-Climate-Model-Results-v1.0.zip'

def save_json(path, value):
    # Reject NaN/Infinity so metadata remains portable, standard JSON.
    path.write_text(json.dumps(value, indent=2, allow_nan=False), encoding='utf-8')

def sha(data):
    return hashlib.sha256(data).hexdigest()

print('Output folder:', ROOT)
print('Forecast: 60 minutes ahead; 72 input readings; CPU; test scoring:', EVALUATE_TEST)

## Cell 2 — Verify the attached results
A **manifest** lists expected files and their fingerprints. Identical attached copies are treated as the same input. Different versions with the same producer label stop the notebook so you can detach extras. Archives are read directly, without unpacking the large camera dataset.

In [ ]:
class Source:
    def __init__(self, path, member=None):
        self.path, self.member = Path(path), member
        if member is None:
            self.manifest_bytes = self.path.read_bytes()
            self.base = self.path.parent
        else:
            with zipfile.ZipFile(self.path) as archive:
                self.manifest_bytes = archive.read(member)
            self.prefix = str(PurePosixPath(member).parent)
        self.manifest = json.loads(self.manifest_bytes)

    def read(self, relative):
        # Accept only paths relative to the producer's artifact directory.
        relative = str(relative).replace(chr(92), '/')
        parts = PurePosixPath(relative)
        assert not parts.is_absolute() and '..' not in parts.parts and ':' not in relative
        if self.member is None:
            target = (self.base / relative).resolve()
            assert target.is_relative_to(self.base.resolve())
            return target.read_bytes()
        name = str(PurePosixPath(self.prefix) / parts)
        with zipfile.ZipFile(self.path) as archive:
            return archive.read(name)

# Discover only manifests belonging to the two required producers.
needed = {'EDEN-01': ('1.1', 'PASS_ETL'),
          'EDEN-04': ('1.0', 'PASS_BASELINE_EVAL_PROVISIONAL')}
candidates = {producer: {} for producer in needed}
def consider(source):
    producer = source.manifest.get('producer')
    if producer in needed:
        if str(source.manifest.get('code_version')) == needed[producer][0]:
            candidates[producer][sha(source.manifest_bytes)] = source

for path in sorted(INPUT.rglob('*.zip')):
    try:
        with zipfile.ZipFile(path) as archive:
            names = [n for n in archive.namelist() if PurePosixPath(n).name == 'manifest.json']
        for name in names:
            consider(Source(path, name))
    except (zipfile.BadZipFile, json.JSONDecodeError, UnicodeDecodeError) as exc:
        print('Skipped unreadable archive metadata:', path, str(exc)[:100])
for path in sorted(INPUT.rglob('manifest.json')):
    try:
        consider(Source(path))
    except (json.JSONDecodeError, UnicodeDecodeError) as exc:
        print('Skipped unreadable metadata:', path, str(exc)[:100])

sources, verification = {}, []
for producer, (version, status) in needed.items():
    choices = list(candidates[producer].values())
    assert len(choices) == 1, f'{producer}: found {len(choices)} distinct manifests. Attach the required result and detach different copies.'
    source = choices[0]
    assert source.manifest['status'] == status, f'{producer}: upstream status is not accepted.'
    files = source.manifest['files']
    assert files and len({f['path'] for f in files}) == len(files)
    for item in files:
        payload = source.read(item['path'])
        assert len(payload) == item['size'] and sha(payload) == item['sha256'], item['path']
    sources[producer] = source
    verification.append(dict(producer=producer, files=len(files), manifest_sha256=sha(source.manifest_bytes)))
    print('VERIFIED', producer, len(files), 'files:', source.path)
source01, source04 = sources['EDEN-01'], sources['EDEN-04']
assert source04.manifest['upstream_manifest_sha256'] == sha(source01.manifest_bytes), 'EDEN-04 was made from a different EDEN-01 result.'
save_json(ROOT / 'reports/input_verification.json', verification)

## Cell 3 — Check time, units, and split labels
The primary protocol holds out whole compartments. The chronological protocol holds out later periods with unused buffers called **embargoes**. We check these separately against EDEN-01's frozen rules. Only observed, QC-approved temperature values can enter the model.

In [ ]:
climate = pd.read_parquet(io.BytesIO(source01.read('data/climate.parquet')))
splits = json.loads(source01.read('data/splits.json'))
schema = json.loads(source01.read('data/schema_map.json'))
required = ['experiment_id','compartment_id','ts_utc','split','chronological_split',
            TARGET, TARGET+'_observed', TARGET+'_qc_ok']
assert set(required) <= set(climate.columns)
climate = climate[required].copy()
climate['compartment_id'] = climate.compartment_id.astype(str)
assert str(climate.ts_utc.dt.tz) == 'UTC'
assert climate.experiment_id.eq(splits['experiment_id']).all()
assert not climate.duplicated(['experiment_id','compartment_id','ts_utc']).any()
assert 1 <= LOOKBACK <= splits['max_lookback_steps']
assert 1 <= HORIZON <= splits['max_horizon_steps']
assert max(LAGS) < LOOKBACK and splits['timestep_minutes'] == 5
metadata = [f for f in schema['fields'] if f['canonical'] == TARGET]
assert metadata and all(f['unit']=='degree_Celsius' and f['kind']=='measurement' for f in metadata)
primary = {cid: role for role, ids in splits['primary']['compartments'].items() for cid in ids}
assert set(climate.compartment_id) == set(primary)
assert climate['split'].eq(climate.compartment_id.map(primary)).all()
c = splits['chronological']
t = climate.ts_utc
expected = np.select([t < pd.Timestamp(c['train_end_exclusive']),
                      (t >= pd.Timestamp(c['validation_start_inclusive'])) & (t < pd.Timestamp(c['validation_end_exclusive'])),
                      t >= pd.Timestamp(c['test_start_inclusive'])],
                     ['train','validation','test'], default='embargo')
assert np.array_equal(climate.chronological_split.to_numpy(), expected)
for cid, group in climate.groupby('compartment_id'):
    assert group.sort_values('ts_utc').ts_utc.diff().dropna().eq(pd.Timedelta(minutes=5)).all(), cid
(ROOT / 'data/splits.json').write_bytes(source01.read('data/splits.json'))
(ROOT / 'LICENSES.txt').write_bytes(source01.read('LICENSES.txt'))
print('Verified climate rows:', len(climate), '| frozen split rules preserved.')

## Cell 4 — Build complete, valid forecast windows
A window contains 72 input readings plus the following 12 ticks to the target. Every tick must stay in one compartment and split. Missing or suspect input readings remove that window; no values are filled. Persistence and Ridge receive exactly the same eligible examples.

In [ ]:
def build_windows(frame, split_column, protocol):
    pieces, audit = [], []
    for cid, original in frame.groupby('compartment_id', sort=True):
        g = original.sort_values('ts_utc').reset_index(drop=True)
        temperature = g[TARGET].astype(float)
        good = (g[TARGET+'_observed'].eq(True) & g[TARGET+'_qc_ok'].eq(True)
                & np.isfinite(temperature))
        # All 72 historical readings must pass QC; only the future target is used as a label.
        history_ok = good.astype(int).rolling(LOOKBACK).sum().eq(LOOKBACK)
        target_ok = good.shift(-HORIZON, fill_value=False)
        base = pd.DataFrame({'experiment_id':g.experiment_id, 'compartment_id':cid,
                             'origin_time':g.ts_utc, 'target_time':g.ts_utc.shift(-HORIZON),
                             'history_start':g.ts_utc.shift(LOOKBACK-1),
                             'actual':temperature.shift(-HORIZON)})
        for lag in LAGS:
            base[f'lag_{lag}'] = temperature.shift(lag)
        base['mean_12'] = temperature.rolling(12).mean()
        base['mean_72'] = temperature.rolling(LOOKBACK).mean()
        # The sum at t+H covers t-(L-1) through t+H, including the forecast interval.
        for role in ('train','validation','test'):
            membership = g[split_column].eq(role).astype(int)
            same_split = membership.rolling(LOOKBACK+HORIZON).sum().shift(-HORIZON).eq(LOOKBACK+HORIZON)
            eligible = same_split & history_ok & target_ok
            keep = base.loc[eligible].copy()
            keep['split'], keep['protocol'] = role, protocol
            audit.append(dict(protocol=protocol, split=role, compartment_id=cid,
                              origin_rows=int(membership.sum()), boundary_eligible=int(same_split.sum()),
                              retained=int(eligible.sum()), qc_excluded=int((same_split & ~eligible).sum())))
            if len(keep):
                assert keep[FEATURES+['actual']].notna().all().all()
                assert (keep.target_time-keep.origin_time).eq(pd.Timedelta(minutes=5*HORIZON)).all()
                assert (keep.origin_time-keep.history_start).eq(pd.Timedelta(minutes=5*(LOOKBACK-1))).all()
                pieces.append(keep)
    assert pieces, f'No usable windows for {protocol}'
    return pd.concat(pieces, ignore_index=True), audit

windows, audit_rows = {}, []
for protocol, column in [('primary','split'), ('chronological','chronological_split')]:
    windows[protocol], rows = build_windows(climate, column, protocol)
    audit_rows.extend(rows)
    counts = windows[protocol].groupby('split').size()
    assert {'train','validation','test'} <= set(counts.index)
    assert counts.min() > 100, f'Too few examples: {protocol}; inspect QC and boundaries.'
    print(protocol, counts.to_dict())
pd.DataFrame(audit_rows).to_csv(ROOT / 'reports/window_coverage.csv', index=False)
save_json(ROOT / 'data/feature_contract.json', dict(target=TARGET, target_unit='degree_Celsius',
          lookback_readings=LOOKBACK, horizon_steps=HORIZON, features=FEATURES,
          protocols=['primary','chronological'], missing_value_policy='exclude_window',
          boundary_policy='all input and intervening target ticks in same split',
          previous_EDEN04_scores_comparable=False))

## Cell 5 — Fit on training and choose on validation
The **scaler** standardizes each input using training statistics. Ridge's **alpha** controls how strongly large weights are discouraged. Four fixed alpha values are compared using validation RMSE. Each protocol gets its own model and scaler; no test scores are used to choose them.

[Scikit-learn's leakage guidance](https://scikit-learn.org/stable/common_pitfalls.html) explains why transformations belong inside a training pipeline. [Ridge documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html) describes the coefficient penalty.

In [ ]:
def errors(actual, predicted):
    delta = np.asarray(predicted)-np.asarray(actual)
    assert len(delta) and np.isfinite(delta).all()
    return dict(n=len(delta), mae_c=float(np.abs(delta).mean()),
                rmse_c=float(np.sqrt(np.mean(delta**2))), bias_c=float(delta.mean()))

models, selection, validation_frames = {}, [], []
for protocol, table in windows.items():
    train = table[table.split.eq('train')]
    valid = table[table.split.eq('validation')]
    candidates = []
    for alpha in ALPHAS:
        pipeline = make_pipeline(StandardScaler(), Ridge(alpha=alpha, solver='svd'))
        # Only training rows are ever passed to fit(), including scaler fitting.
        pipeline.fit(train[FEATURES], train.actual)
        prediction = pipeline.predict(valid[FEATURES])
        stats = errors(valid.actual, prediction)
        selection.append(dict(protocol=protocol, alpha=alpha, **stats))
        candidates.append((stats['rmse_c'], alpha, pipeline))
    _, alpha, chosen = min(candidates, key=lambda item: (item[0], item[1]))
    models[protocol] = chosen
    scaler, ridge = chosen.steps[0][1], chosen.steps[1][1]
    # Export portable numeric parameters, not an executable pickle.
    parameters = dict(protocol=protocol, alpha=alpha, feature_names=FEATURES,
                      scaler_mean=scaler.mean_.tolist(), scaler_scale=scaler.scale_.tolist(),
                      coefficients=ridge.coef_.tolist(), intercept=float(ridge.intercept_),
                      train_n=len(train), validation_n=len(valid),
                      selection_metric='validation_rmse_c', test_used_for_selection=False)
    save_json(ROOT / 'models' / f'{protocol}_ridge.json', parameters)
    # Check the exported numbers reproduce the pipeline's predictions.
    manual = ((valid[FEATURES].to_numpy()-scaler.mean_)/scaler.scale_) @ ridge.coef_ + ridge.intercept_
    assert np.allclose(manual, chosen.predict(valid[FEATURES]), atol=1e-8)
    out = valid[['experiment_id','compartment_id','origin_time','target_time','actual','protocol','split']].copy()
    out['ridge'], out['persistence'] = manual, valid.lag_0.to_numpy()
    validation_frames.append(out)
    print(protocol, '| chosen alpha:', alpha)
pd.DataFrame(selection).to_csv(ROOT / 'reports/validation_model_selection.csv', index=False)
# This record is written before the optional test-scoring cell.
save_json(ROOT / 'models/freeze_record.json', dict(generated_utc=pd.Timestamp.now(tz='UTC').isoformat(),
          models={p:sha((ROOT/'models'/f'{p}_ridge.json').read_bytes()) for p in models},
          selection_uses_validation_only=True, EDEN04_test_summaries_previously_seen=True,
          test_scoring_enabled=EVALUATE_TEST))

## Cell 6 — Report matched errors and day-block uncertainty
Both forecasts are scored on the same rows. A **day-block bootstrap** resamples whole UTC days to show how the mean error difference varies; paired model errors stay together, including compartments sharing weather. The interval is exploratory: validation helped choose alpha, and days are not fully independent. It is not a per-prediction uncertainty interval.

Test scoring stays disabled during development. Enable it only after freezing the model plan; it uses the already selected models without refitting.

In [ ]:
prediction_frames = list(validation_frames)
if EVALUATE_TEST:
    for protocol, table in windows.items():
        test = table[table.split.eq('test')]
        out = test[['experiment_id','compartment_id','origin_time','target_time','actual','protocol','split']].copy()
        out['ridge'] = models[protocol].predict(test[FEATURES])
        out['persistence'] = test.lag_0.to_numpy()
        prediction_frames.append(out)
predictions = pd.concat(prediction_frames, ignore_index=True)
predictions.to_csv(ROOT / 'data/predictions.csv', index=False)
metric_rows, compartment_rows, uncertainty = [], [], []
for (protocol, role), group in predictions.groupby(['protocol','split']):
    for model in ('ridge','persistence'):
        metric_rows.append(dict(protocol=protocol, split=role, model=model, **errors(group.actual, group[model])))
    for cid, part in group.groupby('compartment_id'):
        for model in ('ridge','persistence'):
            compartment_rows.append(dict(protocol=protocol, split=role, compartment_id=cid,
                                         model=model, **errors(part.actual, part[model])))
    # Negative paired differences mean Ridge has lower absolute error on these rows.
    paired = pd.DataFrame({'day':group.target_time.dt.floor('D'),
                           'delta':(group.ridge-group.actual).abs()-(group.persistence-group.actual).abs()})
    blocks = paired.groupby('day').delta.agg(['sum','count'])
    if len(blocks) < 5:
        uncertainty.append(dict(protocol=protocol, split=role, days=len(blocks), status='insufficient_day_blocks'))
        continue
    rng = np.random.default_rng(SEED)
    samples = rng.integers(0, len(blocks), size=(500, len(blocks)))
    draws = blocks['sum'].to_numpy()[samples].sum(axis=1) / blocks['count'].to_numpy()[samples].sum(axis=1)
    lo, hi = np.quantile(draws, [0.025, 0.975])
    uncertainty.append(dict(protocol=protocol, split=role, days=len(blocks),
                            mae_difference_c=float(paired.delta.mean()), low_c=float(lo), high_c=float(hi),
                            status='exploratory_day_block_interval'))
metrics = pd.DataFrame(metric_rows)
metrics.to_csv(ROOT / 'reports/matched_metrics.csv', index=False)
pd.DataFrame(compartment_rows).to_csv(ROOT / 'reports/compartment_metrics.csv', index=False)
pd.DataFrame(uncertainty).to_csv(ROOT / 'reports/error_difference_intervals.csv', index=False)
display(metrics)
print('Negative MAE difference favors Ridge; improvement is not required for a valid run.')

## Cell 7 — Package the result and checks
A successful status means the pipeline ran with valid windows and traceable artifacts. It does not promise the learned model beats persistence. Runtime versions, input hashes, split rules, coefficients, and model-selection scores are included so the run can be checked later.

In [ ]:
assert set(predictions.protocol) == {'primary','chronological'}
assert not predictions.duplicated(['protocol','split','compartment_id','origin_time']).any()
assert EVALUATE_TEST or predictions.split.eq('validation').all()
for (protocol, role), group in metrics.groupby(['protocol','split']):
    assert set(group.model) == {'ridge','persistence'} and group.n.nunique() == 1
freeze = json.loads((ROOT/'models/freeze_record.json').read_text())
for protocol, digest in freeze['models'].items():
    assert sha((ROOT/'models'/f'{protocol}_ridge.json').read_bytes()) == digest
status = 'PASS_MODEL_EVAL_PROVISIONAL' if EVALUATE_TEST else 'PASS_VALIDATION_MODEL_PILOT_PROVISIONAL'
report = dict(status=status, target=TARGET, horizon_minutes=5*HORIZON, input_readings=LOOKBACK,
              validation_only=not EVALUATE_TEST, protocols=['primary','chronological'],
              learned_model='standardized_ridge', comparison='matched_persistence',
              previous_EDEN04_scores_reusable=False,
              previous_EDEN04_issue='Yesterday baseline exceeds six-hour history and can cross chronological splits.',
              scope_status='PROVISIONAL_PENDING_FULL_SRS',
              limitations=['Single temperature target and season; no intervention or control model.',
                           'Validation is used for model selection, so uncertainty intervals are exploratory.',
                           'EDEN-04 test summaries were already viewed; no claim of untouched test.',
                           'Image quality and metric calibration remain unresolved.'])
save_json(ROOT / 'reports/model_report.json', report)
save_json(ROOT / 'reports/environment.json', {'python':platform.python_version(),
          **{p:importlib.metadata.version(p) for p in ['numpy','pandas','pyarrow','scikit-learn']}})
files = []
for path in sorted(ROOT.rglob('*')):
    if path.is_file() and path != ROOT/'manifest.json':
        files.append(dict(path=path.relative_to(ROOT).as_posix(), size=path.stat().st_size, sha256=sha(path.read_bytes())))
manifest = dict(schema_version='1.0', producer='EDEN-05', code_version='1.0', status=status,
                generated_utc=pd.Timestamp.now(tz='UTC').isoformat(),
                scope_status='PROVISIONAL_PENDING_FULL_SRS',
                upstream_manifests={p:sha(s.manifest_bytes) for p,s in sources.items()}, files=files)
save_json(ROOT/'manifest.json', manifest)
# Write to a temporary ZIP, then replace the named output only after a complete export.
temporary = RESULT.with_suffix('.zip.part')
with zipfile.ZipFile(temporary,'w',compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(ROOT.rglob('*')):
        if path.is_file():
            archive.write(path, 'eden/'+path.relative_to(ROOT).as_posix())
with zipfile.ZipFile(temporary) as archive:
    assert archive.testzip() is None
    for item in files:
        payload = archive.read('eden/'+item['path'])
        assert len(payload)==item['size'] and sha(payload)==item['sha256']
temporary.replace(RESULT)
print(status)
print('Download and send back:', RESULT)

## Your Kaggle steps
1. Import this notebook. Select **CPU** and **Internet Off**.
2. Attach **EDEN-01 v1.1 results** and **EDEN-04 v1.0 results**. Your existing dataset can contain both as ZIPs or extracted folders; paths are discovered automatically.
3. Leave `EVALUATE_TEST = False`. Run All, then save the executed version.
4. Download `EDEN-05-Climate-Model-Results-v1.0.zip` from the Output panel and share it for review.

**Troubleshooting:** Missing input: attach the matching producer's result, not only its notebook. Multiple different manifests: detach extra versions. Hash mismatch: reattach the original result. Missing PyArrow/scikit-learn: use a standard Kaggle Python environment with these dependencies. No valid windows: keep the checks intact and share the error and coverage report. If Ridge is worse than persistence, retain that result; do not tune against the test set.

Dataset: Maree et al. (2025), [4TU.ResearchData, v1](https://doi.org/10.4121/fa102772-32db-4b30-bace-12f2016722ce.v1), CC BY 4.0. Source attribution is copied into the result ZIP.